In [ ]:
# Install necessary libraries
!pip install datasets evaluate transformers
!pip install -U datasets huggingface_hub fsspec
!pip install bitsandbytes


In [ ]:
# Import necessary libraries

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
import evaluate


In [3]:
# Set random seeds for reproducibility of your trainings run
def set_seeds(s):
    torch.manual_seed(s)
    np.random.seed(s)
    set_seed(s)

set_seeds(42)

In [4]:
# Load the data
data = pd.read_csv("three_vs_rest.csv")
data = data.rename(columns={"target": "label"})
# Split the data into train and test sets
train_df = data[data["set"]=="train"]
test_df = data[data["set"]=="test"]

train_df = train_df.drop(columns=["set","validation"])
test_df = test_df.drop(columns=["set","validation"])

# Shuffle train_df
train_df = train_df.sample(frac=1, random_state=42)

# Reset the index of the shuffled DataFrame
train_df = train_df.reset_index(drop=True)

# Print the shape of the data
print("Train set size:", train_df.shape[0])
print("Test set size:", test_df.shape[0])
# Print the columns of the data
print("Columns in train set and test set:", train_df.columns.tolist(), test_df.columns.tolist())

Train set size: 2990
Test set size: 5743
Columns in train set and test set: ['sequence', 'label'] ['sequence', 'label']


In [ ]:
## Transfer learning with ESM-2 8M model without fine-tuning for regression task

# Set the model
checkpoint = "facebook/esm2_t6_8M_UR50D"

# load the tokenizer for the ESM-2 model

tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=True)

# Function to tokenize the sequences in the dataset

def tokenize_function(example):
    return tokenizer(example["sequence"], truncation=True)

# Convert the DataFrames to Hugging Face Datasets and tokenize them

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
train_dataset = train_dataset.map(tokenize_function, batched=True)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Train datset:", train_dataset)
print("Test datset:", test_dataset)

# Load the pre-trained model for sequence classification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)

# Freeze all layers except the classification head

for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

# Set up the training arguments

model_name = checkpoint.split("/")[-1]
train_batch_size = 8
eval_batch_size = 8
epochs = 20
seed = 42
lr = 3e-4


args = TrainingArguments(
    model_name,
    warmup_steps=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=lr,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=epochs,
    optim = "paged_adamw_8bit",
    seed = seed,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to = "none",
)

# Function to compute metrics during evaluation
def compute_metrics(eval_pred):
    metric = evaluate.load("spearmanr")
    predictions, labels = eval_pred
    return metric.compute(predictions=predictions, references=labels)


# Create a Trainer instance with the model, training arguments, datasets, tokenizer, and metrics function

trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model using the Trainer instance

trainer.train()

# Save model
model.save_pretrained("esm2_8M_regression_my_model_dir")

# Retrieve the log history from the trainer state
log_history = trainer.state.log_history
df_log = pd.DataFrame(log_history)

# Save the log history to a CSV file
df_log.to_csv("esm2_8M_regression_training_logs.csv", index=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2990 [00:00<?, ? examples/s]

...

Train datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2990
})
Test datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 5743
})


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Spearmanr
1,1.232700,1.671977,0.472691
2,1.173200,1.490583,0.476329
3,1.149900,1.538738,0.482274
4,1.126900,1.589640,0.493132
5,1.114200,1.381412,0.504054
6,1.107000,1.537024,0.510228
7,1.080200,1.436641,0.523054
8,1.082800,1.427492,0.530836
9,1.059300,1.382158,0.539376
10,1.051100,1.523239,0.544774


In [ ]:
## Fine-tuning ESM-2 8M model for regression task

# Set the model
checkpoint = "facebook/esm2_t6_8M_UR50D"

# load the tokenizer for the ESM-2 model

tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=True)

# Function to tokenize the sequences in the dataset

def tokenize_function(example):
    return tokenizer(example["sequence"], truncation=True)

# Convert the DataFrames to Hugging Face Datasets and tokenize them

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
train_dataset = train_dataset.map(tokenize_function, batched=True)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Train datset:", train_dataset)
print("Test datset:", test_dataset)

# Load the pre-trained model for sequence classification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)

# Set up the training arguments

model_name = checkpoint.split("/")[-1]
train_batch_size = 8
eval_batch_size = 8
epochs = 20
seed = 42
lr = 3e-4


args = TrainingArguments(
    model_name,
    warmup_steps=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=lr,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=epochs,
    optim = "paged_adamw_8bit",
    seed = seed,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to = "none",
)



# Function to compute metrics during evaluation
def compute_metrics(eval_pred):
    metric = evaluate.load("spearmanr")
    predictions, labels = eval_pred
    return metric.compute(predictions=predictions, references=labels)


# Create a Trainer instance with the model, training arguments, datasets, tokenizer, and metrics function

trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model using the Trainer instance

trainer.train()

# Save model
model.save_pretrained("esm2_8M_regression_fine_tuned_my_model_dir")

# Retrieve the log history from the trainer state
log_history = trainer.state.log_history
df_log = pd.DataFrame(log_history)

# Save the log history to a CSV file
df_log.to_csv("esm2_8M_regression_fine_tuned_training_logs.csv", index=False)


Map:   0%|          | 0/2990 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/5743 [00:00<?, ? examples/s]

Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Train datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2990
})
Test datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 5743
})


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Spearmanr
1,1.212100,1.914615,0.429031
2,1.015500,1.073498,0.729448
3,0.734700,0.937237,0.780344
4,0.550800,0.869558,0.788992
5,0.476500,0.754793,0.824657
6,0.413800,0.653708,0.842337
7,0.357000,0.634071,0.834735
8,0.312400,0.786658,0.835131
9,0.250800,0.556860,0.846010
10,0.212400,0.548743,0.856744


In [ ]:
## Transfer learning with ESM-2 35M model without fine-tuning for regression task

# Set the model
checkpoint = "facebook/esm2_t12_35M_UR50D"

# load the tokenizer for the ESM-2 model

tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=True)

# Function to tokenize the sequences in the dataset

def tokenize_function(example):
    return tokenizer(example["sequence"], truncation=True)

# Convert the DataFrames to Hugging Face Datasets and tokenize them

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
train_dataset = train_dataset.map(tokenize_function, batched=True)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Train datset:", train_dataset)
print("Test datset:", test_dataset)

# Load the pre-trained model for sequence classification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)

# Freeze all layers except the classification head

for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

# Set up the training arguments

model_name = checkpoint.split("/")[-1]
train_batch_size = 8
eval_batch_size = 8
epochs = 20
seed = 42
lr = 3e-4


args = TrainingArguments(
    model_name,
    warmup_steps=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=lr,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=epochs,
    optim = "paged_adamw_8bit",
    seed = seed,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to = "none",
)


# Function to compute metrics during evaluation
def compute_metrics(eval_pred):
    metric = evaluate.load("spearmanr")
    predictions, labels = eval_pred
    return metric.compute(predictions=predictions, references=labels)


# Create a Trainer instance with the model, training arguments, datasets, tokenizer, and metrics function

trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model using the Trainer instance

trainer.train()

# Save model
model.save_pretrained("esm2_35M_regression_my_model_dir")

# Retrieve the log history from the trainer state
log_history = trainer.state.log_history
df_log = pd.DataFrame(log_history)

# Save the log history to a CSV file
df_log.to_csv("esm2_35M_regression_training_logs.csv", index=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2990 [00:00<?, ? examples/s]

...

Train datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2990
})
Test datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 5743
})


config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/136M [00:00<?, ?B/s]

Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Spearmanr
1,1.214700,1.714876,0.569450
2,1.146200,1.378729,0.578051
3,1.123700,1.374416,0.585762
4,1.094600,1.744668,0.594858
5,1.108700,1.283107,0.601249
6,1.064500,1.494932,0.611144
7,1.058800,1.280791,0.620061
8,1.054900,1.674349,0.627149
9,1.038300,1.295504,0.633405
10,1.010400,1.325144,0.640397


In [ ]:
## Fine-tuning ESM-2 35M model for regression task

# Set the model
checkpoint = "facebook/esm2_t12_35M_UR50D"

# load the tokenizer for the ESM-2 model

tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=True)

# Function to tokenize the sequences in the dataset

def tokenize_function(example):
    return tokenizer(example["sequence"], truncation=True)

# Convert the DataFrames to Hugging Face Datasets and tokenize them

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
train_dataset = train_dataset.map(tokenize_function, batched=True)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Train datset:", train_dataset)
print("Test datset:", test_dataset)

# Load the pre-trained model for sequence classification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)

# Set up the training arguments

model_name = checkpoint.split("/")[-1]
train_batch_size = 8
eval_batch_size = 8
epochs = 20
seed = 42
lr = 3e-4


args = TrainingArguments(
    model_name,
    warmup_steps=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=lr,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=epochs,
    optim = "paged_adamw_8bit",
    seed = seed,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to = "none",
)

# Function to compute metrics during evaluation
def compute_metrics(eval_pred):
    metric = evaluate.load("spearmanr")
    predictions, labels = eval_pred
    return metric.compute(predictions=predictions, references=labels)


# Create a Trainer instance with the model, training arguments, datasets, tokenizer, and metrics function

trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model using the Trainer instance

trainer.train()

# Save model
model.save_pretrained("esm2_35M_regression_fine_tuned_my_model_dir")

# Retrieve the log history from the trainer state
log_history = trainer.state.log_history
df_log = pd.DataFrame(log_history)

# Save the log history to a CSV file
df_log.to_csv("esm2_35M_regression_fine_tuned_training_logs.csv", index=False)


Map:   0%|          | 0/2990 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/5743 [00:00<?, ? examples/s]

Train datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2990
})
Test datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 5743
})


Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t12_35M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Spearmanr
1,1.242900,1.969344,0.466841
2,0.988000,1.078676,0.736613
3,0.762300,0.896195,0.789833
4,0.641400,0.880274,0.790072
5,0.529900,0.697746,0.834430
6,0.456600,0.754875,0.828522
7,0.386700,0.559242,0.847731
8,0.348300,0.584068,0.854797
9,0.302500,0.568631,0.852652
10,0.270600,0.492430,0.858913


In [ ]:
## Transfer learning with ESM-2 150M model without fine-tuning for regression task

# Set the model
checkpoint = "facebook/esm2_t30_150M_UR50D"

# load the tokenizer for the ESM-2 model

tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=True)

# Function to tokenize the sequences in the dataset

def tokenize_function(example):
    return tokenizer(example["sequence"], truncation=True)

# Convert the DataFrames to Hugging Face Datasets and tokenize them

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
train_dataset = train_dataset.map(tokenize_function, batched=True)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Train datset:", train_dataset)
print("Test datset:", test_dataset)

# Load the pre-trained model for sequence classification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)

# Freeze all layers except the classification head

for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

# Set up the training arguments

model_name = checkpoint.split("/")[-1]
train_batch_size = 8
eval_batch_size = 8
epochs = 20
seed = 42
lr = 3e-4


args = TrainingArguments(
    model_name,
    warmup_steps=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=lr,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=epochs,
    optim = "paged_adamw_8bit",
    seed = seed,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to = "none",
)


# Function to compute metrics during evaluation
def compute_metrics(eval_pred):
    metric = evaluate.load("spearmanr")
    predictions, labels = eval_pred
    return metric.compute(predictions=predictions, references=labels)


# Create a Trainer instance with the model, training arguments, datasets, tokenizer, and metrics function

trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model using the Trainer instance

trainer.train()

# Save model
model.save_pretrained("esm2_150M_regression_my_model_dir")

# Retrieve the log history from the trainer state
log_history = trainer.state.log_history
df_log = pd.DataFrame(log_history)

# Save the log history to a CSV file
df_log.to_csv("esm2_150M_regression_training_logs.csv", index=False)


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2990 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


...

Train datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2990
})
Test datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 5743
})


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/595M [00:00<?, ?B/s]

Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Spearmanr
1,1.230400,1.738916,0.548380
2,1.173100,1.457126,0.542395
3,1.147000,1.497288,0.539793
4,1.092100,1.650444,0.543603
5,1.097700,1.346028,0.540929
6,1.067700,1.408885,0.549646
7,1.044000,1.280385,0.552434
8,1.056900,1.463356,0.557204
9,1.033500,1.739698,0.558298
10,1.022600,1.280732,0.561817


In [ ]:
## Fine-tuning ESM-2 150M model for regression task

# Set the model
checkpoint = "facebook/esm2_t30_150M_UR50D"

# load the tokenizer for the ESM-2 model

tokenizer = AutoTokenizer.from_pretrained(checkpoint, clean_up_tokenization_spaces=True)

# Function to tokenize the sequences in the dataset

def tokenize_function(example):
    return tokenizer(example["sequence"], truncation=True)

# Convert the DataFrames to Hugging Face Datasets and tokenize them

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
train_dataset = train_dataset.map(tokenize_function, batched=True)

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Train datset:", train_dataset)
print("Test datset:", test_dataset)

# Load the pre-trained model for sequence classification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)

# Set up the training arguments

model_name = checkpoint.split("/")[-1]
train_batch_size = 8
eval_batch_size = 8
epochs = 20
seed = 42
lr = 3e-4


args = TrainingArguments(
    model_name,
    warmup_steps=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=lr,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=epochs,
    optim = "paged_adamw_8bit",
    seed = seed,
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    report_to = "none",
)

# Function to compute metrics during evaluation
def compute_metrics(eval_pred):
    metric = evaluate.load("spearmanr")
    predictions, labels = eval_pred
    return metric.compute(predictions=predictions, references=labels)


# Create a Trainer instance with the model, training arguments, datasets, tokenizer, and metrics function

trainer = Trainer(
    model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model using the Trainer instance

trainer.train()

# Save model
model.save_pretrained("esm2_150M_regression_fine_tuned_my_model_dir")

# Retrieve the log history from the trainer state
log_history = trainer.state.log_history
df_log = pd.DataFrame(log_history)

# Save the log history to a CSV file
df_log.to_csv("esm2_150M_regression_fine_tuned_training_logs.csv", index=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/2990 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/5743 [00:00<?, ? examples/s]

Train datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2990
})
Test datset: Dataset({
    features: ['sequence', 'label', 'input_ids', 'attention_mask'],
    num_rows: 5743
})


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/595M [00:00<?, ?B/s]

Some weights of EsmForSequenceClassification were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.


Epoch,Training Loss,Validation Loss,Spearmanr
1,1.267000,1.647358,0.324494
2,1.014100,1.048636,0.788244
3,0.701500,0.848650,0.815687
4,0.618400,0.842783,0.797558
5,0.580400,1.185993,0.530516
6,0.553600,0.749112,0.823872
7,0.463900,0.739753,0.833311
8,0.413400,0.640961,0.836510
9,0.346600,0.582769,0.844831
10,0.325400,0.514566,0.863777
